In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Please enable GPU in Colab.")

GPU Available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers datasets accelerate matplotlib tqdm sentencepiece huggingface_hub

In [16]:
%cd /content

!git clone https://github.com/musfiraumar/RLHF-Summarizer-from-scratch.git

%cd /content/RLHF-Summarizer-from-scratch

/content
fatal: destination path 'RLHF-Summarizer-from-scratch' already exists and is not an empty directory.
/content/RLHF-Summarizer-from-scratch


In [6]:
!ls
!ls starter

common.py    README.md	test-common.py	train_reward.py
evaluate.py  results	train_dpo.py	train_sft.py
__pycache__  starter	train_ppo.py
batching.py  check_starter.py  config.py  data.py  models.py  __pycache__


In [7]:
!python starter/check_starter.py

config.json: 100% 704/704 [00:00<00:00, 2.86MB/s]
tokenizer_config.json: 100% 3.66k/3.66k [00:00<00:00, 9.00MB/s]
vocab.json: 100% 801k/801k [00:00<00:00, 19.2MB/s]
merges.txt: 100% 466k/466k [00:00<00:00, 57.7MB/s]
special_tokens_map.json: 100% 831/831 [00:00<00:00, 2.96MB/s]
tokenizer.json: 100% 2.10M/2.10M [00:00<00:00, 76.9MB/s]
README.md: 100% 532/532 [00:00<00:00, 2.48MB/s]

data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:   0% 0.00/111M [00:00<?, ?B/s]s]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:  76% 84.4M/111M [00:02<00:00, 53.4MB/s, 6.88MB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:  82% 90.9M/111M [00:02<00:00, 56.2MB/s, 7.43MB/s  ] ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): downloading bytes:  88% 97.0M/111M [00:02<00:00, 57.3MB/s, 7.96MB/s  ] ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): reconstructing file:  25% 27.4M/111M [00:03<00:06, 12.4MB/s, 1.91MB/s  ]
data/train-00000-of-00001-e8c59e5cf7bce1(…): down

In [22]:
!python train_sft.py

Device: cuda
Loading weights: 100% 272/272 [00:00<00:00, 3690.52it/s]
Baseline validation loss: 2.798371895328208
  0% 1/1250 [00:03<1:08:12,  3.28s/it][W1112 15:02:28.634928340 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 744488960 bytes (free: 6094848, total: 15637086208).
[W1112 15:02:29.401385031 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 744488960 bytes (free: 134021120, total: 15637086208).
  0% 1/1250 [00:04<1:35:47,  4.60s/it]
Traceback (most recent call last):
  File "/content/RLHF-Summarizer-from-scratch/train_sft.py", line 103, in <module>
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/_tensor.py", line 623, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
  

In [21]:
%cd /content/RLHF-Summarizer-from-scratch
!git pull
!ls -lh train_sft.py

/content/RLHF-Summarizer-from-scratch
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 896 bytes | 448.00 KiB/s, done.
From https://github.com/musfiraumar/RLHF-Summarizer-from-scratch
   fb1cfbb..38a9f8a  main       -> origin/main
Updating fb1cfbb..38a9f8a
Fast-forward
 common.py      |  1 +
 notebook.ipynb | 44 ++++++++++++++++++--------------------------
 train_sft.py   |  8 ++++----
 3 files changed, 23 insertions(+), 30 deletions(-)
-rw-r--r-- 1 root root 5.7K Nov 12 15:00 train_sft.py


In [13]:
!ls -lh checkpoints/sft/

ls: cannot access 'checkpoints/sft/': No such file or directory


In [ ]:
import json

with open("results/sft_log.json") as f:
    logs = json.load(f)

print("Baseline Val Loss:", logs["baseline_val_loss"])
print("Final Val Loss:", logs["final_val_loss"])
print("Training Time:", logs["total_training_time_seconds"] / 60, "minutes")
print("GPU:", logs["gpu"])

In [ ]:
!pip install -q -U huggingface_hub

from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="checkpoints/sft",
    repo_id="YOUR_USERNAME/RLHF-Summarizer-SFT",
    repo_type="model"
)

print("SFT model uploaded successfully!")